# Day 21: GenAI Intro and Prompt Engineering
**Transformers, LLM architectures, mixture of experts, and prompting with CoT, ReAct and Tree of Thought**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.

## 1. Setup and API key
With an `OPENAI_API_KEY` the notebook calls a real model. Without one, it runs in **offline demo mode** using a tiny stand-in, so every cell still works and you can study the code.

In [ ]:
# ===== Option A: Google Colab (active) =====
# Add your key once: click the key icon (Secrets) in the left sidebar, name it OPENAI_API_KEY, and allow notebook access.
!pip install -q openai scikit-learn numpy pandas tiktoken

# ===== Option B: VS Code / local =====
# pip install openai scikit-learn numpy pandas tiktoken
# Create a .env file or set the variable in your terminal:  OPENAI_API_KEY=sk-...
# No key? Everything still runs in OFFLINE demo mode with a tiny built-in stand-in model.
print("Setup done")

In [ ]:
%%writefile genai_helpers.py
"""Shared helpers for the GenAI days.
llm()   -> calls OpenAI when a key is available, otherwise a small OFFLINE stand-in so every cell still runs.
embed() -> OpenAI embeddings online, a hashing-based stand-in offline (same API, lower quality).
"""
import json, os, re
import numpy as np

def _get_key():
    try:
        from google.colab import userdata          # Colab Secrets
        return userdata.get("OPENAI_API_KEY")
    except Exception:
        return os.getenv("OPENAI_API_KEY")

OPENAI_API_KEY = _get_key()
ONLINE = bool(OPENAI_API_KEY)
MODEL = os.getenv("LLM_MODEL", "gpt-4o-mini")                 # change to any current chat model you have access to
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")
if ONLINE:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

WORD = re.compile(r"[a-z0-9]+")

def _words(t):
    # crude normalisation so "expires" matches "expire"
    return {w[:-1] if w.endswith("s") and len(w) > 4 else w for w in WORD.findall(t.lower()) if len(w) > 3}

def offline_llm(prompt, system=""):
    """A deliberately simple stand-in. It is NOT intelligent; it lets the code paths run without a key."""
    low = prompt.lower()
    if "context:" in low and "question:" in low:
        ctx = re.split(r"(?i)context:", prompt, 1)[1]
        ctx, q = re.split(r"(?i)question:", ctx, 1)
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", ctx.strip()) if s.strip()]
        best = max(sents, key=lambda s: len(_words(s) & _words(q)), default="")
        return best if best and _words(best) & _words(q) else "I don't know based on the provided context."
    if "classify" in low and "positive" in low:
        neg = sum(w in low for w in ["slow", "bad", "broken", "late", "angry", "worst", "refund"])
        pos = sum(w in low for w in ["great", "love", "fast", "thanks", "excellent", "happy"])
        return "negative" if neg > pos else "positive" if pos > neg else "neutral"
    if "json" in low:
        text = prompt.strip().splitlines()[-1]
        return json.dumps({"summary": " ".join(text.split()[:12]), "offline": True})
    first = re.split(r"(?<=[.!?])\s+", prompt.strip())[0]
    return f"[offline demo] {first[:160]}"

def llm(prompt, system="You are a helpful assistant.", temperature=0.2, model=None):
    if ONLINE:
        r = client.chat.completions.create(
            model=model or MODEL, temperature=temperature,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}])
        return r.choices[0].message.content
    return offline_llm(prompt, system)

def embed(texts):
    """Return an (n, d) array of unit-length vectors."""
    if isinstance(texts, str):
        texts = [texts]
    if ONLINE:
        r = client.embeddings.create(model=EMBED_MODEL, input=texts)
        v = np.array([d.embedding for d in r.data])
    else:
        from sklearn.feature_extraction.text import HashingVectorizer
        hv = HashingVectorizer(n_features=1024, ngram_range=(1, 2), alternate_sign=False, norm=None)
        v = hv.transform([t.lower() for t in texts]).toarray()
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.where(n == 0, 1, n)

print("Mode:", f"ONLINE ({MODEL})" if ONLINE else "OFFLINE demo (add OPENAI_API_KEY to use a real model)")

In [ ]:
import importlib, genai_helpers
importlib.reload(genai_helpers)
from genai_helpers import llm, embed, ONLINE

## 2. Tokens

In [ ]:
texts = {
    "tweet": "Loving the new Task Board release, the filters are so fast!",
    "email": "Hi team, the deployment to production is scheduled for Friday at 6 pm IST. " * 6,
}
try:
    import tiktoken
    enc = tiktoken.get_encoding("o200k_base")
    count = lambda t: len(enc.encode(t))
    print("Example tokens:", [enc.decode([t]) for t in enc.encode("Tokenisation is fascinating")])
except Exception:
    count = lambda t: max(1, round(len(t) / 4))      # rough rule: about 4 characters per token
    print("tiktoken not available, using the 4 characters per token estimate")

for name, t in texts.items():
    print(f"{name:<6} {len(t):>4} chars -> about {count(t)} tokens")

## 3. Next-token prediction and temperature

In [ ]:
import numpy as np
rng = np.random.default_rng(21)
words = ["sunny", "rainy", "cloudy", "windy", "purple"]
logits = np.array([3.0, 1.8, 1.5, 0.8, -1.0])     # the model's raw scores for "Tomorrow will be ..."

def softmax(x, T=1.0):
    z = np.exp((x - x.max()) / T)
    return z / z.sum()

for T in [0.2, 1.0, 2.0]:
    p = softmax(logits, T)
    sample = rng.choice(words, size=8, p=p)
    print(f"T={T}: " + ", ".join(f"{w} {pr:.2f}" for w, pr in zip(words, p)))
    print("      samples:", list(sample))

## 4. Self-attention from scratch

In [ ]:
import pandas as pd
tokens = ["I", "sat", "by", "the", "bank", "of", "the", "river"]
E = {"I": [1, 0, 0, 0], "sat": [0, 1, 0, 0], "by": [0, 0, .2, 0], "the": [0, 0, .1, .1],
     "bank": [0, .2, .9, .8], "of": [0, 0, .1, 0], "river": [0, .1, .8, .9]}
X = np.array([E[t] for t in tokens], dtype=float)

def attention(Q, K, V):
    scores = Q @ K.T / np.sqrt(Q.shape[-1])
    w = np.exp(scores - scores.max(axis=1, keepdims=True))
    w = w / w.sum(axis=1, keepdims=True)
    return w @ V, w

out, w = attention(X, X, X)          # real models learn separate Q, K, V projections
weights = pd.DataFrame(w, index=tokens, columns=tokens).round(2)
print(weights.loc["bank"])
others = weights.loc["bank"].drop("bank")
print("\n'bank' attends most to:", others.idxmax())

## 5. Mixture of experts: a toy router

In [ ]:
experts = ["maths", "code", "language", "facts"]
router = np.array([[2.0, 0.1, 0.2, 0.1], [0.1, 2.2, 0.3, 0.1], [0.1, 0.2, 1.8, 0.6], [0.3, 0.1, 0.4, 2.0]])
token_features = {"integral": [1, 0, 0, .2], "def": [0, 1, .1, 0], "beautiful": [0, 0, 1, .1], "Tokyo": [0, 0, .3, 1]}
for tok, f in token_features.items():
    gate = softmax(np.array(f) @ router)
    top2 = np.argsort(gate)[::-1][:2]
    print(f"{tok:<10} -> experts {[experts[i] for i in top2]} with weights {gate[top2].round(2)}")

## 6. Zero-shot vs few-shot

In [ ]:
messages = ["The app is so slow today", "Love the new dashboard, thanks!", "Refund still not received, worst support",
            "Fast delivery, excellent packaging", "It works", "Login broken again", "Great feature, happy team", "Late again"]
labels = ["negative", "positive", "negative", "positive", "neutral", "negative", "positive", "negative"]

zero = "Classify the sentiment as positive, negative or neutral. Reply with one word.\nMessage: {m}"
few = '''Classify the sentiment as positive, negative or neutral. Reply with one word.
Message: "Checkout was quick and easy" -> positive
Message: "Package arrived damaged" -> negative
Message: "I received the invoice" -> neutral
Message: "{m}" ->'''

for name, tmpl in [("zero-shot", zero), ("few-shot", few)]:
    preds = [llm(tmpl.format(m=m), temperature=0).strip().lower().strip(".") for m in messages]
    acc = sum(p == l for p, l in zip(preds, labels))
    print(f"{name:<9} accuracy {acc}/{len(labels)}  {preds}")
if not ONLINE:
    print("Note: offline mode uses a keyword stand-in, so these scores say nothing about real models. Add a key to compare properly.")

## 7. Chain of Thought

In [ ]:
question = "A plan costs 1,200 per month. Annual billing gives 2 months free. GST is 18%. What is the annual cost with GST?"
cot_prompt = question + "\nThink step by step, then give the final number on the last line as: Answer: <number>"
print(llm(cot_prompt, temperature=0))

# Always verify numbers with code when it matters:
print("\nChecked with Python:", 1200 * 10 * 1.18)

## 8. ReAct: reasoning plus tools

In [ ]:
import re
PRICES = {"P-100": 2499, "P-200": 899}
TOOLS = {
    "lookup_price": lambda arg: str(PRICES.get(arg.strip('"'), "unknown")),
    "calculator": lambda arg: str(round(eval(arg.strip('"'), {"__builtins__": {}}), 2)),   # demo only: never eval untrusted input in production
}

REACT_SYSTEM = '''Answer the question using tools. Use exactly this format:
Thought: <reasoning>
Action: <tool>["<input>"]
After an Observation, continue. When done, write: Final Answer: <answer>
Tools: lookup_price["<product id>"], calculator["<python arithmetic>"]'''

def scripted_policy(transcript):
    # Offline stand-in for the model: a fixed plan, so you can see the loop work without a key.
    if "Observation" not in transcript:
        return 'Thought: I need the price first.\nAction: lookup_price["P-100"]'
    if transcript.count("Observation") == 1:
        price = re.findall(r"Observation: (\S+)", transcript)[-1]
        return f'Thought: Now compute 18% GST.\nAction: calculator["{price} * 0.18"]'
    gst = re.findall(r"Observation: (\S+)", transcript)[-1]
    return f"Thought: I have the GST.\nFinal Answer: The GST is {gst}."

def model_policy(transcript):
    return llm(transcript, system=REACT_SYSTEM, temperature=0)

def react(question, policy, max_steps=6):
    transcript = f"Question: {question}"
    for _ in range(max_steps):
        step = policy(transcript)
        print(step)
        transcript += "\n" + step
        if "Final Answer:" in step:
            return step.split("Final Answer:")[-1].strip()
        m = re.search(r'Action:\s*(\w+)\[(.*)\]', step)
        if not m or m.group(1) not in TOOLS:
            transcript += "\nObservation: unknown tool, use one of " + ", ".join(TOOLS)
            continue
        obs = TOOLS[m.group(1)](m.group(2))
        print("Observation:", obs)
        transcript += f"\nObservation: {obs}"
    return "Stopped: too many steps"

answer = react("What is 18% GST on the price of P-100?", model_policy if ONLINE else scripted_policy)
print("\nANSWER:", answer)

### TODO 1: Add a `convert_currency` tool (INR to JPY with a fixed rate) and ask for the GST in yen.

In [ ]:
# TODO: write your code here

## 9. Tree of Thought: explore, evaluate, choose

In [ ]:
goal = "Reduce average customer support response time from 9 hours to under 2 hours within a month."
if ONLINE:
    plans = [llm(f"Propose plan {i} (different from typical ideas) in 2 sentences for: {goal}", temperature=0.9) for i in "ABC"]
else:
    plans = ["Plan A: hire 5 more agents and add a night shift.",
             "Plan B: add an FAQ assistant for common questions and auto-route tickets by category.",
             "Plan C: rebuild the whole support platform from scratch."]

def score(plan):
    if ONLINE:
        r = llm(f"Score this plan for the goal '{goal}' on impact, effort (10 = easy), risk (10 = safe). "
                f"Reply as JSON with keys impact, effort, risk.\nPlan: {plan}", temperature=0)
        try:
            import json; s = json.loads(r[r.index("{"): r.rindex("}") + 1])
            return {k: float(s[k]) for k in ["impact", "effort", "risk"]}
        except Exception:
            pass
    low = plan.lower()   # offline heuristic scoring
    return {"impact": 8 if "route" in low or "faq" in low else 6, "effort": 3 if "rebuild" in low else 7 if "faq" in low else 5,
            "risk": 2 if "rebuild" in low else 8 if "faq" in low else 6}

rows = []
for p in plans:
    s = score(p)
    rows.append({"plan": p[:70], **s, "total": round(0.5 * s["impact"] + 0.25 * s["effort"] + 0.25 * s["risk"], 2)})
table = pd.DataFrame(rows).sort_values("total", ascending=False)
print(table.to_string(index=False))
print("\nChosen:", table.iloc[0]["plan"])

---
**Ganbatte** (頑張って): *Do your best*

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)